In [18]:
import random
import pandas as pd
import lightgbm as lgb
import joblib
import numpy as np
from sklearn.metrics import recall_score, f1_score, classification_report, make_scorer, fbeta_score
from sklearn.model_selection import ParameterGrid
from sklearn.model_selection import RandomizedSearchCV

In [19]:
print(lgb.__version__)

4.7.0


#### 1. data load

In [20]:
train = pd.read_csv("../data/processed/train.csv")
val = pd.read_csv("../data/processed/val.csv")
test = pd.read_csv("../data/processed/test.csv")

X_train = train.drop(columns=["target"])
y_train = train["target"]
X_val = val.drop(columns=["target"])
y_val = val["target"]

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/train.csv'

#### 2-1. LightGBM training (Recall 0.8456, F1 0.8033)

In [ ]:
model = lgb.LGBMClassifier(
    objective="binary",
    class_weight="balanced",   # 클래스 불균형(32:68) 보정
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="binary_logloss",
    callbacks=[lgb.early_stopping(stopping_rounds=50)] # val set 성능이 더 이상 안 좋아지면 자동으로 멈춰서 과적합 방지
)

print(model)
print('---------------')
print(type(model))

#### 2-2. LightGBM training 하이퍼파라미터 튜닝 (Recall 0.8491, F1 0.7987)

In [ ]:
param_dist = {
    "num_leaves": [15, 31, 50, 70],
    "max_depth": [-1, 5, 8, 12],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [100, 300, 500],
    "min_child_samples": [10, 20, 30],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
}

search = RandomizedSearchCV(
    lgb.LGBMClassifier(objective="binary", class_weight="balanced", random_state=42),
    param_distributions=param_dist,
    n_iter=30,
    scoring="recall",   # 우리가 중요하게 보는 지표 기준으로 탐색
    cv=3,
    random_state=42,
    n_jobs=1
)
search.fit(X_train, y_train)
print(search.best_params_)
model = search.best_estimator_

#### 2-3. 2-2에서 n_iter=30->80 변경 (Recall 0.8596, F1 0.8046) -> 이 모델로 결정
- 2-2. 30번만 무작위로 뽑아서 테스트 / 2-3. 80번으로 시도 횟수를 늘림
- 무작위로 80개의 세트를 뽑아서 시험

In [ ]:
param_dist = {
    "num_leaves": [15, 31, 50, 70],
    "max_depth": [-1, 5, 8, 12],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [100, 300, 500],
    "min_child_samples": [10, 20, 30],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
}

print("Hyperparameter 80iter start")

search = RandomizedSearchCV(
    lgb.LGBMClassifier(objective="binary", class_weight="balanced", random_state=42),
    param_distributions=param_dist,
    n_iter=80,
    scoring="recall",   # Recall 중심 탐색
    cv=3,
    random_state=42,
    n_jobs=1
)
search.fit(X_train, y_train)
print(f"optimal hyperparameter: {search.best_params_}")
model = search.best_estimator_

#### 2-1 ~ 2-3. 평가 (Recall, F1)

In [ ]:
y_pred = model.predict(X_val)

recall = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)

print(f"Recall: {recall:.4f}, F1: {f1:.4f}")
print(classification_report(y_val, y_pred))

#### 2-4. n_iter=80 + threshold(임계값)=0.35 (Recall 0.9018, F1 0.7581) 

In [ ]:
param_dist = {
    "num_leaves": [15, 31, 50, 70],
    "max_depth": [-1, 5, 8, 12],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [100, 300, 500],
    "min_child_samples": [10, 20, 30],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
}

print("Hyperparameter 80iter start")

search = RandomizedSearchCV(
    lgb.LGBMClassifier(objective="binary", class_weight="balanced", random_state=42),
    param_distributions=param_dist,
    n_iter=80,
    scoring="recall",   # Recall 중심 탐색
    cv=3,
    random_state=42,
    n_jobs=1
)
search.fit(X_train, y_train)
print(f"optimal hyperparameter: {search.best_params_}")
model = search.best_estimator_

# Threshold = 0.35 
threshold = 0.35
print(f"{threshold} 적용 평가 result")

y_proba = model.predict_proba(X_val)[:, 1]   # 자퇴(1)일 확률 ➡️ predict_proba(X_val)는 각 학생이 자퇴할 확률을 0~1 사이 숫자로 줌, [:, 1]은 그 중 클래스 1(자퇴)에 해당하는 확률 열만 뽑음
y_pred = (y_proba >= threshold).astype(int)   # 확률이 0.35 이상이면 True(=1), 미만이면 False(=0)로 바꿈 # .astype(int)로 True/False를 1/0 숫자로 변환

recall = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)

In [ ]:
# 2-4 평가
print(f"Recall: {recall:.4f}, F1: {f1:.4f}")
print(classification_report(y_val, y_pred))

#### 2-5. n_iter=80 + 탐색 기준 지표(scoring) 바꾸기 ->F2 Score + Threshold = 0.35 (Recall 0.9018, F1 0.7581, F2 0.8382)   

In [ ]:
# 1. F2 Scorer 정의

f2_scorer = make_scorer(fbeta_score, beta=2) # Recall에 2배 가중치를 부여하는 F2 score 감싸기

param_dist = {
    "num_leaves": [15, 31, 50, 70],
    "max_depth": [-1, 5, 8, 12],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [100, 300, 500],
    "min_child_samples": [10, 20, 30],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
}

print("F2-Score 기준 Hyperparameter 80iter start")


# 2. F2 Scorer 기반 RandomizedSearchCV 실행

search_f2 = RandomizedSearchCV(
    lgb.LGBMClassifier(objective="binary", class_weight="balanced", random_state=42),
    param_distributions=param_dist,
    n_iter=80,
    scoring=f2_scorer,   # F2 score 기준 탐색
    cv=3,
    random_state=42,
    n_jobs=1
)

search_f2.fit(X_train, y_train)

print(f"optimal hyperparameter: {search_f2.best_params_}")
best_model_f2 = search_f2.best_estimator_


# 3. Threshold = 0.35 적용 + 평가

threshold = 0.35
print(f"{threshold} 적용 평가 result")

# 자퇴(1) 확률 추출
y_pred_proba = best_model_f2.predict_proba(X_val)[:, 1]

# 0.35 기준 클래스 분류
y_pred_custom = (y_pred_proba >= threshold).astype(int)

# 지표 계산
final_recall = recall_score(y_val, y_pred_custom)
final_f1 = f1_score(y_val, y_pred_custom)
final_f2 = fbeta_score(y_val, y_pred_custom, beta=2)

print("-" * 50)
print(f"Recall : {final_recall:.4f}")
print(f"F1     : {final_f1:.4f}")
print(f"F2     : {final_f2:.4f}")

#### 4. result save

In [ ]:
final_model = search.best_estimator_  # n_iter=80, scoring="recall" 튜닝 결과

joblib.dump(final_model, "../models/lightgbm.joblib")

# feature importance
importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": final_model.feature_importances_
}).sort_values("importance", ascending=False)
importance_df.to_csv("../reports/lightgbm_importance.csv", index=False)

### 5. reports/model_results.csv 작성하기

In [ ]:
result_row = {
    "model": "LightGBM",
    "team_member": "조현주",
    "threshold": 0.5,
    "recall": recall_score(y_val, final_model.predict(X_val)),
    "f1": f1_score(y_val, final_model.predict(X_val))
}

result_df = pd.DataFrame([result_row])
result_df.to_csv("../reports/model_results.csv", index=False)
print("model_results.csv 생성 완료")
print(result_df)